# 12. Looking ahead: inspecting a dataset (milestone 2): exercises

Exercises of [chapter 12](../../12_inspecting.md): a preview of milestone 2, made only from what the catalogue holds. Run the setup cell first.

Work in a copy of this notebook, under [`docs/manual/solutions/`](..), in a folder `solutions_<you>`: it is regenerated by [`docs/manual/tools/build_notebooks.py`](../../tools/build_notebooks.py), and its links are written for that folder.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import json
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac, rasterio
from shapely.geometry import shape
from data_acquisition.catalog import build, root, stac
from data_acquisition.catalog import summary as summaries
from data_acquisition.steps import curate


def product(item):
    """The product of an item: the key of its data asset."""
    return next(k for k, a in item.assets.items() if "data" in (a.roles or []))

**1. A table of items.** Build, curate and root the sample (as in [chapter 7](../../07_pipeline_at_a_glance.md)), open `<sample root>/stac/catalog.json` with pystac, and make one table row per item: id, product, kind, site, start and end of its time, date precision, number of flags.

> **From the chapter.** §2: each view reads fields the catalogue already holds. [Chapter 10](../../10_building_and_curating.md) explains build, curate and the root catalogue.
>
> **Code.** `summaries.load`, `build.build`, `curate.curate_item`, `stac.save` and `root.build_root`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py), [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py), [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py), [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py) and [`data_acquisition/catalog/root.py`](../../../../data_acquisition/catalog/root.py). `product(item)` is defined above; `pystac` reads the catalogue.

In [ ]:
s = summaries.load(str(ROOT / "docs/manual/solutions/training_pond.yaml"))
catalog, _ = build.build(SAMPLE, s)
for item in catalog.get_items(recursive=True):
    curate.curate_item(item, s)
stac.save(catalog, SAMPLE / "stac/training_pond")
root.build_root(SAMPLE / "stac")
...  # your code here (pystac.Catalog.from_file, item.datetime, item.properties)

**2. Coverage.** Turn the table into a GeoDataFrame and plot the footprints in the site's UTM zone, coloured by product. What does each footprint cover, and why is the photos' one invisible?

> **From the chapter.** §2, the row on the coverage map: footprints are in longitude and latitude.
>
> **Code.** No pipeline code: `geopandas`, on your table from exercise 1.

In [ ]:
...  # your code here (gpd.GeoDataFrame(table, crs="EPSG:4326"), .to_crs, .plot)

**3. Timeline.** One row per product, a mark per item from its start to its end. Which item is an interval, and why?

> **From the chapter.** §2: a timeline must draw intervals, not only points.
>
> **Code.** No pipeline code: `matplotlib`, on your table from exercise 1.

In [ ]:
...  # your code here (plt.subplots, ax.plot or ax.barh)

**4. Quick-look.** Without a stored quick-look, read the orthomosaic at about 200 pixels wide (`out_shape`) and show it next to the item's flags.

> **From the chapter.** §2: no item stores a quick-look, so the browser must make one with a reduced read.
>
> **Code.** No pipeline code: `rasterio` with `out_shape`. `item.assets[...].get_absolute_href()` gives the file's path.

In [ ]:
item = next(i for i in items if product(i) == "orthomosaic")
...  # your code here (rasterio.open(item.assets[...].get_absolute_href()), read(out_shape=...))

**5. What is missing.** By code: which items have a `thumbnail` asset, which have colonies, and what does `sites.json` say is still to decide?

> **From the chapter.** §3: what the catalogue does not have yet.
>
> **Code.** The items' assets and `beaver:colonies`, and `SAMPLE / "stac/sites.json"`, written by `root.build_root` in [`data_acquisition/catalog/root.py`](../../../../data_acquisition/catalog/root.py).

In [ ]:
...  # your code here